# To do
- ~~Analyze lick data~~
    - ~~RT~~
    - ~~Lick rate~~
    - ~~ILI~~
- History kernels
- ~~Add HMM~~
- Repeating bias per ILD

In [1]:
# Standard
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import matplotlib.ticker as mticker
import seaborn as sns
from scipy.stats import ttest_ind, ttest_rel

# Own
from glue_sessions.glue_sessions import *
from intersession import glue_animals_intersessions
from my_fun import *
from licks.licks import *
from psychometric_curves import *
from kernels.psychophysical_kernels import *
from full_model_behavior import *
from plotting_style import *

In [2]:
# Magics
%load_ext autoreload
%autoreload 2
# %matplotlib inline
%config InlineBackend.figure_format = 'svg'
# %config InlineBackend.figure_format = 'retina'

In [3]:
# Plotting style
# sns.set_theme()
# sns.set_style('ticks')
# sns.set_context('poster')
style_path = os.path.expanduser('~/PycharmProjects/alexis_style.mplstyle')
plt.style.use(style_path)
# default_figsize = np.array(plt.rcParams['figure.figsize'])

sns.set_style('ticks')
sns.set_context('notebook')

# Analyses of drug data from batch #6

# Load behavioral data

In [4]:
# Set parameters
experiment = '2AFC_6'  # Drug

# Load trial data (lick analyses)
df_behavior = glue_animals(experiment=experiment, path_session='glue_sessions', filter_drug=True, update=False, to_csv=False)  # Raw
# df_behavior = glue_animals(experiment=experiment, path_session='glmhmm', filter_drug=True, update=False, to_csv=False)  # GLM HMM fits

# Load intersession data
df_intersessions = glue_animals_intersessions(experiment=experiment, filter_drug=True, update=False, to_csv=False)
df_intersessions['AbsLatBias'] = abs(df_intersessions.LateralBias)  # Add new column

# Filter data

In [5]:
animals = cherry_pick(df_behavior, experiment, plot=False)
df_behavior = df_behavior[df_behavior.Task == 'FD'].reset_index(drop=True)
df_behavior = df_behavior[df_behavior.P > 0].reset_index(drop=True)

# # Something is off in the parsing of the subjects in this group (probably because their IDs start with 0) and their IDs are missing the 0 padding. Add it so everything works
# df_behavior.Subject = df_behavior.Subject.astype(str)
# df_behavior['Subject'] = df_behavior['Subject'].str.zfill(3)
# df_intersessions.Subject = df_intersessions.Subject.astype(str)
# df_intersessions['Subject'] = df_intersessions['Subject'].str.zfill(3)
#
# # Remove subjects not in cherries
# df_behavior = df_behavior[df_behavior.Subject.isin(animals)].reset_index(drop=True)
# df_intersessions = df_intersessions[df_intersessions.Subject.isin(animals)].reset_index(drop=True)
# assert df_behavior.Subject.unique().tolist() == animals

Number of responded trials in data collection (with evidences):
14: 4277 trials
16: 5850 trials
17: 6533 trials
18: 3016 trials
19: 2479 trials
20: 7268 trials
22: 5208 trials
23: 5415 trials
24: 5563 trials
25: 5439 trials


Subjects left behind (never learnt):


Bad subjects (based on lapses): []

Good subjects:
014: 0.46 lapses
016: 0.48 lapses
017: 0.46 lapses
018: 0.61 lapses
019: 0.66 lapses
020: 0.26 lapses
022: 0.42 lapses
023: 0.47 lapses
024: 0.46 lapses
025: 0.42 lapses




In [ ]:
# Remove bad sessions

# Add paired session index
n_paired_sessions = len(df_intersessions) // 2
print(f'Total: {n_paired_sessions:.0f} paired sessions')
paired_sessions = np.repeat(np.arange(n_paired_sessions), 2)
df_intersessions['PairedSession'] = paired_sessions

threshold = 0.5
mask = df_intersessions.Accuracy < threshold
# mask = ((df_intersessions.AccuracyLeft < threshold) | (df_intersessions.AccuracyRight < threshold))

# Remove from intersession data
bad_sessions = df_intersessions[mask].PairedSession
bad_sessions = bad_sessions.unique()
print(f'{len(bad_sessions)} bad paired sessions. Pair indexes: {bad_sessions}')
print(df_intersessions.loc[df_intersessions.PairedSession.isin(bad_sessions), ['PairedSession', 'Accuracy', 'AccuracyLeft', 'AccuracyRight']])
df_intersessions = df_intersessions[~df_intersessions.PairedSession.isin(bad_sessions)].reset_index(drop=True)
print(f'Dropped {(len(bad_sessions) / n_paired_sessions)*100}% paired sessions')

# # Remove from trial data (merging with already filtered intersession data)
# df_behavior['Date'] = pd.to_datetime(df_behavior['Date'])
# df_behavior = df_behavior.merge(
#     df_intersessions[['Subject', 'Dates', 'PairedSession']],
#     left_on=['Subject', 'Date'],
#     right_on=['Subject', 'Dates'],
#     how='left'
# )
#
# # Optionally, drop the extra 'Date' column from df_intersession after merging
# df_behavior = df_behavior.drop(columns='Dates')

In [ ]:
n_subjects = df_behavior.Subject.nunique()
n_paired_sessions = len(df_intersessions) // 2
n_trials = len(df_behavior)
print(f'N subjects = {n_subjects}')
print(f'N paired sessions = {n_paired_sessions}')
print(f'N trials = {n_trials} (saline: {n_trials_saline}, drug: {n_trials_drug}')

# Behavioral metrics

## Standard

In [ ]:
# Define function to compare behavioral metrics between saline and drug conditions
def plot_var_drug(df_intersessions, var_name='Accuracy', **kwargs):
    """
    Plot a variable (e.g., accuracy) for each subject and drug condition together with the mean and sem across subjects, and run a t-test between saline and drug conditions.
    :param df_intersessions: DataFrame containing intersession data from all animals
    :param var_name: Name of the variable to plot (e.g., 'Accuracy')
    :return: None
    """

    var = df_intersessions.groupby(['Subject', 'Drug'])[var_name].mean().reset_index()
    var_mean = var.groupby('Drug')[var_name].mean()
    var_sem = var.groupby('Drug')[var_name].sem()

    # Plot paired lines
    plt.figure(constrained_layout=True, **kwargs)
    sns.boxplot(data=var, x='Drug', y=var_name, palette=['tab:gray', 'tab:pink'], showfliers=False)
    sns.lineplot(data=var, x='Drug', y=var_name, hue='Subject', marker='', palette=['gray'] * var.Subject.nunique(), alpha=0.25, legend=False)
    # Plot mean and SEM
    # plt.errorbar(x=var_mean.index, y=var_mean.values, yerr=var_sem.values, fmt='-o', color=color)
    # plt.title(var_name)
    plt.xticks([0, 1], ['Saline', 'Drug'])
    plt.xlabel('')
    # plt.ylim([0.5, 1])
    plt.ylabel(var_name)
    # plt.gca().yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f'))  # Set a global formatter for y-axis ticks for equal sized plots
    sns.despine()

    # Run t-test
    saline = var[var['Drug'] == 0][var_name]
    drug = var[var['Drug'] == 1][var_name]
    t_stat, p_val = ttest_rel(saline, drug)
    print(f'{var_name}: t = {t_stat:.3f}, p = {p_val:.3f}')
    add_star_between(p_val)

In [ ]:
# figsize = fig_size(n_cols=3)
# plot_var_drug(df_intersessions, var_name='Accuracy', figsize=figsize)
# plot_var_drug(df_intersessions, var_name='AccMaxEvi', figsize=figsize)
# plot_var_drug(df_intersessions, var_name='AccNonMaxEvi', figsize=figsize)
# plot_var_drug(df_intersessions, var_name='LateralBias', figsize=figsize)
# plot_var_drug(df_intersessions, var_name='AbsLatBias', figsize=figsize)
# plot_var_drug(df_intersessions, var_name='MissRate', figsize=figsize)
# plot_var_drug(df_intersessions, var_name='CorrRepBias', figsize=figsize)

## Licks

### Add lick data

In [ ]:
# Save old columns for comparison later
old_columns = list(df_behavior.columns)

# Add lick data to the DataFrame
df_behavior = add_lick_data(df_behavior)

# Add session half index (0 for the 1st and 1 for the 2nd)
df_behavior['SessionHalf'] = (df_behavior.Trial >= df_behavior.groupby('Session').Trial.transform('max') / 2).astype(int)
# Add absolute ILD
loc = df_behavior.columns.get_loc('ILD') + 1
df_behavior.insert(loc, 'absILD', df_behavior['ILD'].abs())

# Print new columns added to the DataFrame
new_columns = list(df_behavior.columns)
new_columns = [col for col in new_columns if col not in old_columns]
print(new_columns)

# df_behavior.head()

### RTs

In [ ]:
plot_licks_split(df_behavior, var='RT', split='drug', kind='kde')

In [ ]:
# plot_licks_per_subject(df_behavior, plot_func=plot_licks_split, var='RT', split='drug', kind='kde')

In [ ]:
# Stats
plot_var_drug(df_behavior, var_name='RT')
# plot_var_drug(df_behavior[df_behavior.Hit==0], var_name='RT', color='tab:red')
# plot_var_drug(df_behavior[df_behavior.Hit==1], var_name='RT', color='tab:green')

In [ ]:
# Saline
plt.figure(figsize=default_figsize/2, constrained_layout=True)
plot_ild_dist_mean(df_behavior[df_behavior.Drug==0], var='RT')
plt.title('Saline')

In [ ]:
# Drug
plt.figure(figsize=default_figsize/2, constrained_layout=True)
plot_ild_dist_mean(df_behavior[df_behavior.Drug==1], var='RT')
plt.title('Drug')

### Lick rate

In [ ]:
# Correct trials only (that's why the number of trials is half as in RT or ILI distributions)
plot_licks_split(df_behavior[df_behavior.Hit==1], var='nLicks', split='drug', kind='kde')

In [ ]:
# plot_licks_per_subject(df_behavior, plot_func=plot_licks_split, var='nLicks', split='drug', kind='kde')

In [ ]:
plot_var_drug(df_behavior, var_name='nLicks')
# plot_var_drug(df_behavior[df_behavior.Hit==0], var_name='nLicks', color='tab:red')
# plot_var_drug(df_behavior[df_behavior.Hit==1], var_name='nLicks', color='tab:green')

In [ ]:
# Saline
plt.figure(figsize=default_figsize/2, constrained_layout=True)
plot_ild_dist_mean(df_behavior[df_behavior.Drug==0], var='nLicks')
plt.title('Saline')

In [ ]:
# Drug
plt.figure(figsize=default_figsize/2, constrained_layout=True)
plot_ild_dist_mean(df_behavior[df_behavior.Drug==1], var='nLicks')
plt.title('Drug')

### ILI

In [ ]:
plot_licks_split(df_behavior, var='ILI', split='drug', kind='kde')
plt.xlim(0, 0.2)

In [ ]:
# plot_licks_per_subject(df_behavior, plot_func=plot_licks_split, var='ILI', split='drug', kind='kde')

In [ ]:
plot_var_drug(df_behavior, var_name='ILI')
# plot_var_drug(df_behavior[df_behavior.Hit==0], var_name='ILI', color='tab:red')
# plot_var_drug(df_behavior[df_behavior.Hit==1], var_name='ILI', color='tab:green')

In [ ]:
# Saline
plt.figure(figsize=default_figsize/2, constrained_layout=True)
# plot_ild_dist_mean(df_behavior[df_behavior.Drug==0], var='ILI')
plot_ild_dist_mean(df_behavior[(df_behavior.Drug==0) & (df_behavior.Hit==1)], var='ILI')
plt.title('Saline')

In [ ]:
# Drug
plt.figure(figsize=default_figsize/2, constrained_layout=True)
# plot_ild_dist_mean(df_behavior[df_behavior.Drug==1], var='ILI')
plot_ild_dist_mean(df_behavior[(df_behavior.Drug==1) & (df_behavior.Hit==1)], var='ILI')
plt.title('Drug')

#### Mixed Effects Model
To show that the drug removes the ILD dependence you have to show that the interaction is significant. Do a Linear Mixed model of the ILI1 with norm_trial_index, norm_trial_indx ^2, ILD, drug, drug x ILD, curren_outcome, prev_outcome, Left-Right response (binary variable to account for differences in the port side). Put session as the random effect. Do it for each mouse and plot the stats of the weights. Let’s see what comes out significant. You can use the same design matrix for the Nlicks and the z_tranformed_RT.

In [ ]:
import pandas as pd
import statsmodels.formula.api as smf

subjects = df_behavior.Subject.unique()

# Prepare containers
all_fe = []
all_pvals = []

for subj in subjects:

    df_mouse = df_behavior[df_behavior.Subject == subj].copy()
    # df_mouse = df_mouse[df_mouse.Hit == 1]  # Only correct trials

    # Drop absILD==70
    # df_mouse = df_mouse[df_mouse.absILD != 70]

    # Drop NaNs and reset index
    df_mouse = df_mouse.dropna(subset=['ILI','absILD','Drug','Hit','AfterHit','Choice'])
    df_mouse.reset_index(drop=True, inplace=True)

    # df_mouse['absILD'] = df_mouse['absILD'].astype('category')
    # df_mouse['absILD'] = df_mouse['absILD'] - df_mouse['absILD'].mean()
    df_mouse['absILD_c'] = df_mouse['absILD'] / df_mouse['absILD'].max()

    # Add normalized trial index per session
    df_mouse['NormTrial'] = df_mouse.groupby('Session')['Trial'].transform(
        lambda x: (x - x.min()) / (x.max() - x.min()))
    df_mouse['NormTrial2'] = df_mouse['NormTrial'] ** 2  # Squared term

    # Define formula
    formula = 'ILI ~ absILD + Drug + Drug:absILD'

    # # Updated formula
    # formula = ('ILI ~ NormTrial + NormTrial2 + absILD + Drug + '
    #            'Drug:absILD + AfterHit + Choice')

    # Fit model
    model = smf.mixedlm(formula, df_mouse, groups=df_mouse['Session'])
    # model = smf.ols(formula, df_mouse)
    result = model.fit()
    print(result.summary())

    # Store fixed-effect estimates and p-values
    fe = result.fe_params  # For mixedlm
    # fe = result.params  # For ols
    fe.name = subj
    all_fe.append(fe)

    pvals = result.pvalues
    pvals.name = subj
    all_pvals.append(pvals)

# Convert to DataFrames
df_fe = pd.DataFrame(all_fe)
df_p = pd.DataFrame(all_pvals)

# print(df_fe.head())
# print(df_p.head())

In [ ]:
# Compute mean and SEM across mice
fe_mean = df_fe.mean()
fe_sem = df_fe.sem()
p_mean = df_p.mean()
print(fe_mean)
print(fe_sem)
print(p_mean)

# Highlight effects where mean p-value < 0.05
colors = ['red' if df_p[col].mean() < 0.05 else 'gray' for col in df_fe.columns]

plt.figure(constrained_layout=True)
plt.bar(range(len(fe_mean)), fe_mean, yerr=fe_sem, color=colors)
plt.xticks(range(len(fe_mean)), fe_mean.index, rotation=45, ha='right')
plt.axhline(0, color='black', linestyle='--')
plt.ylabel('FE estimate\n(mean ± SEM)')
plt.title('LMM weights')
plt.tight_layout()

# Psychometrics

In [ ]:
# Identify outliers per condition (same as Seaborn)
def remove_outliers(df, group_col, value_col):
    cleaned = []
    for name, group in df.groupby(group_col):
        Q1 = group[value_col].quantile(0.25)
        Q3 = group[value_col].quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR
        cleaned.append(group[(group[value_col] >= lower) & (group[value_col] <= upper)])
    return pd.concat(cleaned, axis=0)
def compare_psych_params(params, param_name, **kwargs):
    """
    Compare the parameters of the saline and drug conditions.
    """

    subject = np.tile(np.arange(len(params) // 2), 2)
    params['subject'] = subject
    var_mean = params.groupby('drug')[param_name].mean()
    var_sem = params.groupby('drug')[param_name].sem()

    plt.figure(constrained_layout=True, **kwargs)

    # Plot boxplot
    sns.boxplot(data=params, x='drug', y=param_name, hue='drug', palette=['tab:gray', 'tab:pink'], showfliers=False, legend=False)
    # sns.swarmplot(data=params, x='drug', y=param_name, color='k', legend=False)

    # Plot paired lines
    params_no_outliers = remove_outliers(params, 'drug', param_name)  # Clean outliers
    sns.lineplot(data=params_no_outliers, x='drug', y=param_name, hue='subject', marker='o', palette=['gray'] * int(len(params)/2), alpha=0.25, legend=False)
    # Plot mean and SEM
    # plt.errorbar(x=var_mean.index, y=var_mean.values, yerr=var_sem.values, fmt='-o', color='k')

    # plt.title(var_name)
    plt.ylabel(param_name)
    plt.gca().yaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f'))  # Set a global formatter for y-axis ticks for equal sized plots
    plt.xticks([0, 1], ['Saline', 'Drug'])
    plt.xlabel('')
    sns.despine()

    # Add stats
    saline = params[params['drug'] == 0][param_name]
    drug = params[params['drug'] == 1][param_name]
    t_stat, p_val = ttest_rel(saline, drug)
    print(f"{param_name}: t = {t_stat:.3f}, p = {p_val:.3f}")
    add_star_between(p_val)

## Probability choose RIGHT

In [ ]:
kind = 'prob_right'
figsize = fig_size(n_cols=2)
figsize = (figsize[0], figsize[0])

### Single animals

In [ ]:
# for animal in animals:
#     print(f'Mouse {animal}')
#     plot_pc_drug(experiment='2AFC_6', animal=animal, kind=kind, figsize=figsize)

### Mean across animals

In [ ]:
# params = plot_mean_pc_drug(experiment='2AFC_6', animals=animals, kind=kind, figsize=figsize)

#### Stats

In [ ]:
# figsize = fig_size(n_cols=4)
# params['bias'] = abs(params['bias'])  # Use absolute bias
# compare_psych_params(params, 'sensitivity', figsize=figsize)
# compare_psych_params(params, 'bias', figsize=figsize)
# compare_psych_params(params, 'lr_right', figsize=figsize)
# compare_psych_params(params, 'lr_left', figsize=figsize)
# params['lapses'] = (params['lr_right'] + params['lr_left'])  # Total lapse rates
# compare_psych_params(params, 'lapses', figsize=figsize)

## Probability choose REPEAT

In [ ]:
kind = 'prob_rep'
figsize = fig_size(n_cols=2)
figsize = (figsize[0], figsize[0])

### Single animals

In [ ]:
# for animal in animals:
#     print(f'Mouse {animal}')
#     plot_pc_drug(experiment='2AFC_6', animal=animal, kind=kind, figsize=figsize)

### Mean across animals

In [ ]:
# params = plot_mean_pc_drug(experiment='2AFC_6', animals=animals, kind=kind, figsize=figsize)

#### Stats

In [ ]:
# figsize = fig_size(n_cols=4)
# params['bias'] = abs(params['bias'])  # Use absolute bias
# compare_psych_params(params, 'sensitivity', figsize=figsize)
# compare_psych_params(params, 'bias', figsize=figsize)
# compare_psych_params(params, 'lr_rep', figsize=figsize)
# compare_psych_params(params, 'lr_alt', figsize=figsize)
# params['lapses'] = (params['lr_rep'] + params['lr_alt'])  # Total lapse rates
# compare_psych_params(params, 'lapses', figsize=figsize)

# Kernels
Full GLM of behavior

In [ ]:
# Parameters
residuals = True
zscore = False
drug = None
trial_lag = 10
iterations = 1
save = False
figsize = fig_size(n_cols=2)

experiments = ['2AFC_6']
# hk = get_mean_hk(experiments=experiments, animals=animals, drug=None, trial_lag=trial_lag, iterations=iterations)
# plot_hk(experiment=experiments, animal=None, drug=None, trial_lag=trial_lag, iterations=iterations, save=save, figsize=figsize)

In [ ]:
# Example subject
figsize = fig_size(n_cols=2)
plot_drug_hk(experiment='2AFC_6', animal='016', drug=None, trial_lag=trial_lag, iterations=iterations, save=save, figsize=figsize)

In [ ]:
# # Mean all mice
# figsize = fig_size(n_cols=2)
# plot_drug_hk(experiment=experiments, animal=None, drug=None, trial_lag=trial_lag, iterations=iterations, save=save, figsize=figsize)

# GLM HMM

In [ ]:
# Make a new column 'Engaged' that is 1 if state is 0 and is 0 if state is 1 or 2
engaged = [1 if state == 0 else 0 for state in df_behavior.State]
df_behavior['Engaged'] = engaged
df_engaged = df_behavior[df_behavior.Engaged == 1]
df_disengaged = df_behavior[df_behavior.Engaged == 0]

In [ ]:
# Fraction of engagement
plot_var_drug(df_behavior, var_name='Engaged')

## Licks

In [ ]:
def plot_ild_dist_mean_eng_drug(var='RT'):

    # Define conditions and titles
    conditions = [
        (0, 0),  # Disengaged - Saline
        (0, 1),  # Disengaged - Drug
        (1, 0),  # Engaged - Saline
        (1, 1)  # Engaged - Drug
    ]

    # Compute global y-limits
    all_means = []
    for engaged, drug in conditions:
        df = df_behavior[(df_behavior.Engaged==engaged) & (df_behavior.Drug==drug)]
        for ild in df_behavior.absILD.unique():
            all_means.append(df[df.absILD==ild][var].mean())
    y_min, y_max = min(all_means), max(all_means)
    y_ticks = np.linspace(y_min, y_max, 3)

    # Create 2x2 subplots
    fig, axes = plt.subplots(2, 2, figsize=default_figsize, constrained_layout=True)

    # Loop over conditions
    for engaged, drug in conditions:
        ax = axes[engaged, drug]
        plt.sca(ax)
        plot_ild_dist_mean(df_behavior[(df_behavior.Engaged == engaged) & (df_behavior.Drug == drug)], var=var)
        plt.ylim(None, y_max)
        # plt.yticks(y_ticks)

    # Set row labels
    axes[0, 0].set_ylabel('Disengaged')
    axes[1, 0].set_ylabel('Engaged')

    # Set column labels
    axes[0, 0].set_title('Saline')
    axes[0, 1].set_title('Drug')

    # Remove xlabels in 1st row
    axes[0, 0].set_xlabel('')
    axes[0, 1].set_xlabel('')

    # Remove titles in 2nd row
    axes[1, 0].set_title('')
    axes[1, 1].set_title('')

    # Remove ylabels in 2nd column
    axes[0, 1].set_ylabel('')
    axes[1, 1].set_ylabel('')

    plt.suptitle('Mean ' + var)

In [ ]:
def test(var='RT'):
    abs_ilds = sorted(df_behavior.absILD.unique())
    x = np.arange(len(abs_ilds))
    conditions = [
        (0, 0, 'tab:gray', '--', 'Dis.-Sal.'),
        # (0, 1, 'tab:pink', '--', 'Dis.-Drug'),
        (1, 0, 'tab:gray', '-', 'Eng.-Sal.'),
        # (1, 1, 'tab:pink', '-', 'Eng.-Drug')
    ]

    plt.figure(figsize=default_figsize)
    for engaged, drug, color, ls, label in conditions:
        df = df_behavior[(df_behavior.Engaged==engaged) & (df_behavior.Drug==drug)]
        means = [df[df.absILD==ild][var].mean() for ild in abs_ilds]
        sems = [df[df.absILD==ild][var].sem() for ild in abs_ilds]
        plt.errorbar(x, means, yerr=sems, marker='o', color=color, ls=ls, label=label)

    plt.xticks(x, [int(ild) for ild in abs_ilds])
    plt.xlabel('|ILD|')
    plt.ylabel(var)
    plt.legend()
    sns.despine()

### RTs

In [ ]:
plot_ild_dist_mean_eng_drug(var='RT')
test(var='RT')

In [ ]:
plot_ild_dist_mean_eng_drug(var='nLicks')
test(var='nLicks')

In [ ]:
plot_ild_dist_mean_eng_drug(var='ILI')
test(var='ILI')

# Serial dependence

Serial dependence requires a continuous readout of behavioral responses. As choices are binary (left/right), other readouts are needed. Let's try with reaction times (RT) and lick rate (nLicks)

In [ ]:
df = df_behavior

loc = df.columns.get_loc('Side') + 1
df.insert(loc, 'PreviousSide', df['Side'].shift(1))  # Add previous stimulus side column
loc = df.columns.get_loc('PreviousSide') + 1
df.insert(loc, 'RepSide', (df['Side'] == df['PreviousSide']).astype(int))  # Add repeating stimulus column
loc = df.columns.get_loc('ILD') + 1
df.insert(loc, 'PreviousILD', df['ILD'].shift(1))  # Add previous stimulus side column
loc = df.columns.get_loc('PreviousILD') + 1
df.insert(loc, 'DeltaILD', df.PreviousILD - df.ILD)  # Add previous stimulus side column
loc = df.columns.get_loc('DeltaILD') + 1
df.insert(loc, 'absDeltaILD', abs(df.DeltaILD))  # Add previous stimulus side column

# Filter data
df = df[df.Task == 'FD'].reset_index(drop=True)
df = df[df.P > 0].reset_index(drop=True)
df = df[df.Hit == 1].reset_index(drop=True)

# loc = df.columns.get_loc('nLicksRight') + 1  # Added to add_licks function
# df.insert(loc, 'nLicks', np.where(df['Side'] == 0, df['nLicksLeft'], df['nLicksRight']))  # Add number of licks column

# Center at 0 variables of interest
df.nLicks = df.nLicks - df.nLicks.mean()  # Substract its mean to nLicks
df.RT = df.RT - df.RT.mean()  # Substract its mean to RT
df.RT2 = df.RT2 - df.RT2.mean()  # Substract its mean to RT2

df.head()

In [ ]:
# Check value range of DeltaILD for binning later
np.array(sorted(df.DeltaILD.unique()))

In [ ]:
def plot_binned_deltaILD(var='RT', absolute=False):
    """
    Plot binned deltaILD distribution.
    :param var: variable to plot
    :param abs: if True, plot absolute values of deltaILD
    :return:
    """

    # Set bin edges
    if absolute:
        bins = [0, 5, 10, 20, 70, 80, 140]
        col_name = 'absDeltaILD'
        xlabel = 'abs(ΔILD)'
    else:
        bins = [-140, -80, -70, -20, -10, -5 , 5, 10, 20, 70, 80, 140]
        xlabel = 'ΔILD'
        col_name = 'DeltaILD'

    # Create bin labels ( as the bin spacing is not continuous)
    labels = [f"{bins[i]}:{bins[i+1]}" for i in range(len(bins)-1)]

    # Bin the data
    df['ILDbin'] = pd.cut(df[col_name], bins=bins, labels=labels, include_lowest=True)

    # Group and compute means
    mean_licks = df.groupby('ILDbin', as_index=False)[var].mean()

    # Plot
    plt.figure(constrained_layout=True)
    plt.bar(mean_licks['ILDbin'], mean_licks[var], color='tab:gray')
    plt.xlabel(xlabel)
    plt.ylabel(var)
    plt.xticks(rotation=45)
    plt.title(f'{var} by {xlabel}')
    sns.despine()

### Measured as Reaction Times (RTs)

In [ ]:
# Plot mean RT split by RepSide
plt.figure(constrained_layout=True)
sns.kdeplot(df[df.RepSide==0].RT, label='rep_stim', color='tab:purple')
sns.kdeplot(df[df.RepSide==1].RT, label='alt_stim', color='tab:brown')
plt.title('RT distribution')
# plt.legend(frameon=False)
sns.despine()

# Plot means in inset plot
ax = plt.gca()  # get current axes
ax_inset = inset_axes(ax, width='30%', height='30%', loc='upper right')
sns.pointplot(data=df, x='RepSide', y='RT', hue='RepSide', markers='o', palette=['tab:purple', 'tab:brown'], linestyles='', legend=False)
plt.xticks([0, 1], ['Alt', 'Rep'])
plt.xlabel('Stim')

# Run t-test
var_name = 'RT'
alt_stim = df[df['RepSide'] == 0][var_name]
rep_stim = df[df['RepSide'] == 1][var_name]
t_stat, p_val = ttest_ind(alt_stim, rep_stim)
print(f"{var_name}: t-statistic = {t_stat:.3f}, p-value = {p_val:.3f}")
add_star_between(p_val)
sns.despine()

#### ΔILD

In [ ]:
# Plot the RT by Δ change in ILD (previous-current)
plt.figure(constrained_layout=True)
plt.plot(df.groupby('DeltaILD').RT.mean(), marker='o', color='tab:gray')
plt.xlabel('ΔILD')
plt.ylabel('RT')
plt.title('RT by ΔILD')
sns.despine()

plot_binned_deltaILD(var='RT', absolute=False)

##### Absolute ΔILD

In [ ]:
# Plot the RT by absolute Δ change in ILD (previous-current)
plt.figure(constrained_layout=True)
plt.plot(df.groupby('absDeltaILD').RT.mean(), marker='o', color='tab:gray')
plt.xlabel('abs(ΔILD)')
plt.ylabel('RT')
plt.title('RT by abs(ΔILD)')
sns.despine()

plot_binned_deltaILD(var='RT', absolute=True)

### Measured as N licks

In [ ]:
# Plot mean RT split by RepSide
plt.figure(constrained_layout=True)
sns.kdeplot(df[df.RepSide==0].nLicks, label='rep_stim', color='tab:purple')
sns.kdeplot(df[df.RepSide==1].nLicks, label='alt_stim', color='tab:brown')
plt.title('N licks distribution')
# plt.legend(frameon=False)

# Plot means in inset plot
ax = plt.gca()  # get current axes
ax_inset = inset_axes(ax, width='30%', height='30%', loc='upper right')
sns.pointplot(data=df, x='RepSide', y='nLicks', hue='RepSide', markers='o', palette=['tab:purple', 'tab:brown'], linestyles='', legend=False)
plt.xticks([0, 1], ['Alt', 'Rep'])
plt.xlabel('Stim')
sns.despine()

# Run t-test
var_name = 'nLicks'
alt_stim = df[df['RepSide'] == 0][var_name]
rep_stim = df[df['RepSide'] == 1][var_name]
t_stat, p_val = ttest_ind(alt_stim, rep_stim)
print(f"{var_name}: t-statistic = {t_stat:.3f}, p-value = {p_val:.3f}")
add_star_between(p_val)
sns.despine()

#### ΔILD

In [ ]:
# Plot the number of licks by Δ change in ILD (previous-current)
plt.figure(constrained_layout=True)
plt.plot(df.groupby('DeltaILD').nLicks.mean(), marker='o', color='tab:gray')
plt.xlabel('ΔILD')
plt.ylabel('nLicks')
plt.title('nLicks by ΔILD')
sns.despine()

plot_binned_deltaILD(var='nLicks', absolute=False)

##### Absolute ΔILD

In [ ]:
# Plot the number of licks by absolute Δ change in ILD (previous-current)
plt.figure(constrained_layout=True)
plt.plot(df.groupby('absDeltaILD').nLicks.mean(), marker='o', color='tab:gray')
plt.xlabel('abs(ΔILD)')
plt.ylabel('nLicks')
plt.title('nLicks by abs(ΔILD)')
sns.despine()

plot_binned_deltaILD(var='nLicks', absolute=True)

# Save all notebook's figures

In [ ]:
# # Whole notebook ouputs (no input or prompt, only Markdown and figures)
# # As HTML
# !jupyter nbconvert notebook.ipynb --to html --no-input --no-prompt
# # As PDF
# !jupyter nbconvert notebook.ipynb --to pdf --no-input --no-prompt

# Export all figures
# save_notebook_files('notebook.ipynb')